# WaveRNN vocoder training

Trains WaveRNN on CLArTTS-style parquet splits using `commons.dataset.WaveRNNDataset` and `WaveRNNConfig` from `commons.hyperparams`.

After each epoch, outputs go to **`WaveRNNConfig.monitor_dir`** (default `wavernn/training_monitor/`): **`loss_curve.png`** (train/val loss) and **`audio/epoch_XXXX.wav`** (vocoder sample from a fixed validation mel so you can hear progress).

The next cell finds the repo root (directory containing `commons/hyperparams.py`) so imports work regardless of the kernel working directory.

In [ ]:
import os
import sys
from pathlib import Path


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "commons" / "hyperparams.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()


REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
import csv
import glob
import os
import time

import matplotlib.pyplot as plt
import torch
import torchaudio
import torch.nn.functional as F
from torch.utils.data import DataLoader

from commons.hyperparams import (
    WaveRNNConfig,
    Tacotron2Config,
    WAVERNN_TRAIN_PATH,
    WAVERNN_VAL_PATH,
    WAVERNN_TEST_PATH,
)
from commons.dataset import WaveRNNDataset, denormalize, normalize
from wavernn.wavernn import WaveRNN
from tacotron2.model import Tacotron2
from tacotron2.tokenizer import Tokenizer

In [ ]:
def save_checkpoint(model, optimizer, epoch, step, avg_loss, config):
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    tagged = os.path.join(config.checkpoint_dir, f"wavernn_epoch{epoch:04d}.pt")
    state = {
        "epoch": epoch,
        "step": step,
        "avg_loss": avg_loss,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(state, tagged)
    torch.save(state, os.path.join(config.checkpoint_dir, config.checkpoint_name))

    all_ckpts = sorted(glob.glob(os.path.join(config.checkpoint_dir, "wavernn_epoch*.pt")))
    for old in all_ckpts[:-5]:
        os.remove(old)
    print(f"  [CKPT] Saved -> {tagged}")


def load_checkpoint(model, optimizer, config, device):
    path = config.resume_from or os.path.join(config.checkpoint_dir, config.checkpoint_name)
    if not os.path.exists(path):
        print("  [CKPT] No checkpoint found – starting from scratch.")
        return 0, 0
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    epoch = ckpt.get("epoch", 0)
    step = ckpt.get("step", 0)
    print(f"  [CKPT] Resumed from epoch {epoch}, step {step} | {path}")
    return epoch, step


def build_model(config) -> WaveRNN:
    product = 1
    for s in config.upsample_scales:
        product *= s
    assert product == config.hop_length, (
        f"product(upsample_scales) = {product} != hop_length = {config.hop_length}. "
        f"Fix WaveRNNConfig.upsample_scales."
    )
    model = WaveRNN(
        upsample_scales=list(config.upsample_scales),
        n_classes=config.n_classes,
        hop_length=config.hop_length,
        n_res_block=config.n_res_block,
        n_rnn=config.n_rnn,
        n_fc=config.n_fc,
        kernel_size=config.kernel_size,
        n_freq=config.n_mels,
        n_hidden=config.n_hidden,
        n_output=config.n_output,
    )
    return model


@torch.no_grad()
def validate(model, val_loader, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    n = 0
    for i, (mel_seg, waveform_in, target) in enumerate(val_loader):
        if max_batches and i >= max_batches:
            break
        mel_seg = mel_seg.to(device)
        waveform_in = waveform_in.unsqueeze(1).to(device)
        target = target.to(device)
        specgram = mel_seg.unsqueeze(1)

        output = model(waveform_in, specgram).squeeze(1)
        B, T, C = output.shape
        loss = F.cross_entropy(output.reshape(B * T, C), target.reshape(B * T))
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(n, 1)


def ensure_monitor_dirs(config):
    os.makedirs(config.monitor_dir, exist_ok=True)
    os.makedirs(os.path.join(config.monitor_dir, "audio"), exist_ok=True)


def load_loss_history(log_path):
    epochs, train_l, val_l = [], [], []
    if not os.path.exists(log_path):
        return epochs, train_l, val_l
    with open(log_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row["epoch"]))
            train_l.append(float(row["train_loss"]))
            val_l.append(float(row["val_loss"]))
    return epochs, train_l, val_l


def save_loss_plot(epochs, train_losses, val_losses, out_path):
    if not epochs:
        return
    plt.figure(figsize=(9, 4))
    plt.plot(epochs, train_losses, label="train", color="#2a6f97")
    plt.plot(epochs, val_losses, label="val", color="#e76f51")
    plt.xlabel("epoch")
    plt.ylabel("cross-entropy loss")
    plt.title("WaveRNN training")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


@torch.no_grad()
def save_sample_wav(model, mel_batch, path, sample_rate, device):
    """Write first item in batch as WAV (mel_batch: [B, n_mels, T])."""
    model.eval()
    mel_batch = mel_batch.to(device)
    wav, _ = model.infer(mel_batch)
    model.train()
    w = wav[0].cpu().clamp(-1, 1)
    torchaudio.save(str(path), w, sample_rate)


def show_monitor_outputs(loss_png, wav_path=None):
    """Show loss figure (and optionally play audio) in the notebook when IPython is available."""
    try:
        from IPython.display import Audio, Image, display

        display(Image(filename=str(loss_png)))
        if wav_path is not None and os.path.isfile(wav_path):
            display(Audio(filename=str(wav_path)))
    except Exception:
        pass

In [ ]:
def train():
    config = WaveRNNConfig()
    torch.manual_seed(config.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 60)
    print("  WaveRNN Training")
    print(f"  Device      : {device}")
    print(f"  Train path  : {WAVERNN_TRAIN_PATH}")
    print(f"  Val path    : {WAVERNN_VAL_PATH}")
    print(f"  Test path   : {WAVERNN_TEST_PATH}  (not used in this loop)")
    print(f"  Monitor dir : {config.monitor_dir}")
    print(f"  Batch size  : {config.batch_size}")
    print(f"  LR          : {config.learning_rate}")
    print(f"  Epochs      : {config.epochs}")
    print(f"  Seg frames  : {config.segment_mel_frames}")
    print(f"  n_mels      : {config.n_mels}")
    print(
        f"  upsample    : {config.upsample_scales} -> product={eval('*'.join(str(s) for s in config.upsample_scales))}"
    )
    print("=" * 60)

    train_ds = WaveRNNDataset(WAVERNN_TRAIN_PATH, config)
    val_ds = WaveRNNDataset(WAVERNN_VAL_PATH, config)

    train_loader = DataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

    print(f"  Train: {len(train_ds):,} samples | {len(train_loader)} batches/epoch")
    print(f"  Val  : {len(val_ds):,} samples   | {len(val_loader)} batches")

    # Pick a fixed non-silent mel from the val set for per-epoch monitoring.
    # Scan batches until we find a segment whose mean absolute value exceeds a
    # minimum threshold (mels normalised to [-4, 4]; near-silence sits near -4).
    _MEL_SILENCE_THRESHOLD = 0.1  # mean |mel| below this is considered silent/padding
    fixed_mel = None
    for _batch in val_loader:
        _mels = _batch[0]  # [B, n_mels, T]
        for _i in range(_mels.size(0)):
            if _mels[_i].abs().mean().item() > _MEL_SILENCE_THRESHOLD:
                fixed_mel = _mels[_i : _i + 1].contiguous()
                break
        if fixed_mel is not None:
            break
    if fixed_mel is None:
        raise RuntimeError(
            "Could not find a non-silent mel segment in the validation set. "
            "Check that the parquet audio is correctly loaded and normalised."
        )
    print(f"  [monitor] fixed_mel mean|val|={fixed_mel.abs().mean().item():.3f}  shape={tuple(fixed_mel.shape)}")

    model = build_model(config).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  WaveRNN parameters: {n_params:,}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    start_epoch, global_step = load_checkpoint(model, optimizer, config, device)

    os.makedirs(config.checkpoint_dir, exist_ok=True)
    ensure_monitor_dirs(config)
    log_path = os.path.join(config.checkpoint_dir, "loss_log.csv")
    if not os.path.exists(log_path):
        with open(log_path, "w") as f:
            f.write("epoch,train_loss,val_loss\n")

    loss_png = os.path.join(config.monitor_dir, "loss_curve.png")

    for epoch in range(start_epoch + 1, config.epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_start = time.time()

        for batch_idx, (mel_seg, waveform_in, target) in enumerate(train_loader):
            mel_seg = mel_seg.to(device)
            waveform_in = waveform_in.unsqueeze(1).to(device)
            target = target.to(device)
            specgram = mel_seg.unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)

            output = model(waveform_in, specgram).squeeze(1)
            B, T, C = output.shape
            loss = F.cross_entropy(
                output.reshape(B * T, C),
                target.reshape(B * T),
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()

            epoch_loss += loss.item()
            global_step += 1

            if batch_idx % 100 == 0:
                elapsed = time.time() - epoch_start
                print(
                    f"  Ep {epoch:03d} | Batch {batch_idx:04d}/{len(train_loader)} "
                    f"| Loss: {loss.item():.4f} | Step: {global_step} | {elapsed:.0f}s"
                )

        avg_train_loss = epoch_loss / len(train_loader)

        save_checkpoint(model, optimizer, epoch, global_step, avg_train_loss, config)

        avg_val_loss = validate(model, val_loader, device, max_batches=config.eval_batches)

        elapsed_total = time.time() - epoch_start
        print(
            f"\n[EPOCH {epoch:03d}] train_loss={avg_train_loss:.4f} | "
            f"val_loss={avg_val_loss:.4f} | time={elapsed_total:.0f}s\n"
        )

        with open(log_path, "a") as f:
            f.write(f"{epoch},{avg_train_loss:.6f},{avg_val_loss:.6f}\n")

        epoch_list, train_losses, val_losses = load_loss_history(log_path)
        save_loss_plot(epoch_list, train_losses, val_losses, loss_png)

        wav_path = os.path.join(config.monitor_dir, "audio", f"epoch_{epoch:04d}.wav")
        save_sample_wav(model, fixed_mel, wav_path, config.sample_rate, device)
        print(f"  [monitor] {loss_png} | {wav_path}")

        show_monitor_outputs(loss_png, wav_path)

    print("[DONE] WaveRNN training complete.")

In [ ]:
train()